In [ ]:
# pip install -q xgboost==2.0.3 optuna imbalanced-learn

Model Training

In [ ]:
# Default Prediction with Monotonic Risk Banding


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
import optuna
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("/content/drive/MyDrive/fyp/data/new_cleaned_dataset.csv")
#df = pd.read_csv("data/new_cleaned_dataset.csv")
target_column = "CONTRACT_STATUS"

X = df.drop(columns=[target_column])
y = df[target_column]

# feature engineering
X['LTI'] = X['FINANCE_AMOUNT'] / (X['INCOME'] + 1e-5)
X['DTI'] = (X['EXPENSE'] + X['FINANCE_AMOUNT']) / (X['INCOME'] + 1e-5)
X['Disposable_Income'] = X['INCOME'] - X['EXPENSE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# One-hot encode categoricals
cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# Optuna tuning
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "eval_metric": "logloss",
        "use_label_encoder": False,
        "tree_method": "gpu_hist",      #using GPU training
        "predictor": "gpu_predictor",
        "n_jobs": -1,
    }

    model = XGBClassifier(**params)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []

    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        ros = RandomOverSampler(random_state=42)
        X_tr_res, y_tr_res = ros.fit_resample(X_tr, y_tr)

        model.fit(X_tr_res, y_tr_res)
        y_pred_prob = model.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, y_pred_prob))

    return np.mean(aucs)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best ROC-AUC (CV):", round(study.best_value, 4))
print("Best Parameters:", study.best_params)

# Train final model 
best_params = study.best_params
final_model = XGBClassifier(
    **best_params,
    random_state=42,
    eval_metric="logloss",
    use_label_encoder=False,
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    n_jobs=-1,
)

ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)
final_model.fit(X_train_res, y_train_res)

# Evaluate on test 
y_pred_prob = final_model.predict_proba(X_test)[:, 1]
roc_auc_test = roc_auc_score(y_test, y_pred_prob)
print("Test ROC-AUC:", round(roc_auc_test, 4))

# ROC curve 
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"ROC (AUC={roc_auc_test:.3f})", color="blue", lw=3)
plt.plot([0,1], [0,1], "r--", lw=2, label="Random")
plt.fill_between(fpr, tpr, alpha=0.3)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Monotonic Risk-Band assignment 
results = pd.DataFrame({
    "Default_Actual": y_test.values,
    "Default_Prob":  y_pred_prob
})

# Sort by predicted probability and split evenly into 7 groups
results = results.sort_values("Default_Prob").reset_index(drop=True)
results["Risk_Band"] = pd.qcut(
    results.index, q=7,
    labels=[
        "Band_1_Excellent",
        "Band_2_Very_Good",
        "Band_3_Good",
        "Band_4_Acceptable",
        "Band_5_Moderate",
        "Band_6_Elevated",
        "Band_7_High_Risk"
    ]
)

band_stats = (
    results.groupby("Risk_Band")
    .agg(Customers=("Default_Actual", "count"),
         Default_Rate=("Default_Actual", "mean"))
    .round(4)
    .reset_index()
)
print("\nBusiness Bands Stats:\n", band_stats)

# Visualize monotonic pattern 
plt.figure(figsize=(10,6))
plt.bar(band_stats["Risk_Band"], band_stats["Default_Rate"],
        color="skyblue", edgecolor="black")
plt.ylabel("Default Rate")
plt.title("Default Rate by Risk Band (Monotonic)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.show()


Save Trained Model.

default_risk_model.pkl → Trained model.

model_features.pkl → Feature list for alignment during inference

In [ ]:
import joblib

# Save model
joblib.dump(final_model, "/content/drive/MyDrive/fyp/trained/default_risk_model.pkl")

# Optionally save supporting objects
joblib.dump(X_train.columns, "/content/drive/MyDrive/fyp/trained/model_features.pkl")


Use of saved model

In [ ]:
import joblib
import pandas as pd

# Load model and feature list
model = joblib.load("/content/drive/MyDrive/fyp/trained/default_risk_model.pkl")
model_features = joblib.load("/content/drive/MyDrive/fyp/trained/model_features.pkl")

# new input 
new_data = pd.DataFrame([
    {
        "FINANCE_AMOUNT": 290000,
        "INCOME": 46000,
        "EXPENSE": 15000,
        "DUE_FREQUENCY": 0,
        "NET_RENTAL": 11996,
        "NO_OF_RENTAL": 36,
        "CUSTOMER_VALUATION": 375000,
        "EFFECTIVE_RATE": 28,
        "AGE": 62,
        "MARITAL_STATUS": "M"
    }
])

# feature engineering
new_data["LTI"] = new_data["FINANCE_AMOUNT"] / (new_data["INCOME"] + 1e-5)
new_data["DTI"] = (new_data["EXPENSE"] + new_data["FINANCE_AMOUNT"]) / (new_data["INCOME"] + 1e-5)
new_data["Disposable_Income"] = new_data["INCOME"] - new_data["EXPENSE"]

# Align with training columns
new_data = pd.get_dummies(new_data)
new_data = new_data.reindex(columns=model_features, fill_value=0)

# Predict default probability
prob = model.predict_proba(new_data)[:, 1][0]
print("Predicted Default Probability:", prob)
